# CP via Class Similarity – Demo

End-to-end walkthrough of all score families and penalty variants.

### Package layout
```
src/
  config.py                       # global hyperparameters (edit here)
  data_loader.py                  # load logits, adjacency & similarity matrices
  score_functions.py              # RAPS / LAC / SAPS score + prediction-set builders
  metrics.py                      # coverage / set-size evaluation helpers
  experiments_original.py         # RAPS paper-replication runners
  experiments_tuned_lambda_strict_split.py  # RAPS strict-split runners
  experiments_lac.py              # LAC runners (original + strict_split)
  experiments_saps.py             # SAPS runners (original + strict_split)
```

### Original vs strict_split
Functions in `experiments_*.py` that end in `_strict_split` keep **cal / val / test**
strictly separate (exchangeability-safe).  The plain versions follow the paper's
original protocol and are provided for exact replication.


In [1]:
import sys
sys.path.insert(0, '..')   # make src/ importable from notebooks/

%load_ext autoreload
%autoreload 2

import numpy as np
import src.config as config
import src.metrics as metrics
import src.score_functions as score_functions
import src.data_loader as data_loader
import src.experiments_original as experiments_original
import src.experiments_tuned_lambda_strict_split as experiments_strict_split
import src.experiments_lac as experiments_lac
import src.experiments_saps as experiments_saps


## 1. Load data

In [2]:
# Edit these paths to point at your .npz files
LOGITS_PATH     = '../data/Cifar100_ResNet50_logits_new.npz'
ADJACENCY_PATH  = '../data/Adjacency_matrix.npz'
SIMILARITY_PATH = '../data/similarity_matrix_cosine_resnet50.npz'

probabilities, labels = data_loader.load_logits(LOGITS_PATH)
adjacency_matrix, adjacency_matrix_smaller = data_loader.load_adjacency_matrix(ADJACENCY_PATH)
adjacency_matrix_ms = data_loader.load_similarity_matrix_cosine(SIMILARITY_PATH)

print('probabilities:', probabilities.shape)
print('labels:       ', labels.shape)
print('adjacency:    ', adjacency_matrix.shape)

probabilities: (10000, 100)
labels:        (10000,)
adjacency:     (100, 101)


## 2. RAPS experiments

In [3]:
# ── Baseline RAPS (no class-similarity penalty) ───────────────────────────
results_raps_base = experiments_original.run_raps_baseline(probabilities, labels)

Marginal coverage            : 0.9002
Mean class-conditional cov.  : 0.9002
Worst-class CC deviation     : 0.1345  (class 35)
Top-5 worst CC deviation     : 0.1175
Avg set size                 : 2.693  (std 3.950, min 0, max 27)
Avg distinct superclasses    : 0.000


In [4]:
lamda_values = [0, 0.01, 0.03, 0.06, 0.1, 0.15, 0.2, 0.3]

# ── RAPS + MA superclass-mass penalty (original protocol) ─────────────────
results_raps_ma_orig = experiments_original.run_raps_ma_avg_opt(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_smaller,
    lamda_values,
)

Marginal coverage            : 0.9011
Mean class-conditional cov.  : 0.9011
Worst-class CC deviation     : 0.1294  (class 78)
Top-5 worst CC deviation     : 0.1115
Avg set size                 : 1.955  (std 2.073, min 0, max 21)
Avg distinct superclasses    : 1.321


In [5]:
# ── RAPS + MA superclass-mass penalty (strict_split) ─────────────────────────
results_raps_ma_corr = experiments_strict_split.run_raps_ma_avg_opt_strict_split(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_smaller,
    lamda_values,
)

Marginal coverage            : 0.9014
Mean class-conditional cov.  : 0.9014
Worst-class CC deviation     : 0.1273  (class 78)
Top-5 worst CC deviation     : 0.1140
Avg set size                 : 1.974  (std 2.126, min 0, max 21)
Avg distinct superclasses    : 1.333


In [6]:
# ── RAPS + MS binary-distance penalty (strict_split) ─────────────────────────
results_raps_ms_corr = experiments_strict_split.run_raps_ms_avg_opt_strict_split(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_ms,
    lamda_values,
)

# ── Optional fixed-lambda variant for the binary-distance penalty ─────────────
# This skips validation-based lambda selection and uses a fixed lambda directly.
results_raps_ms_fixed_lambda = experiments_original.run_raps_ms_avg_opt(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_ms,
    [0.0, 0.5],
    fixed_lambda=0.25,
)


Marginal coverage            : 0.9004
Mean class-conditional cov.  : 0.9004
Worst-class CC deviation     : 0.1565  (class 10)
Top-5 worst CC deviation     : 0.1163
Avg set size                 : 1.856  (std 2.014, min 1, max 19)
Avg distinct superclasses    : 1.372
Marginal coverage            : 0.9017
Mean class-conditional cov.  : 0.9018
Worst-class CC deviation     : 0.1489  (class 10)
Top-5 worst CC deviation     : 0.1102
Avg set size                 : 1.908  (std 2.119, min 1, max 19)
Avg distinct superclasses    : 1.398


## 3. LAC experiments

In [7]:
# ── Baseline LAC ──────────────────────────────────────────────────────────
results_lac_base = experiments_lac.run_lac_baseline(probabilities, labels)

Marginal coverage            : 0.9037
Mean class-conditional cov.  : 0.9037
Worst-class CC deviation     : 0.1796  (class 35)
Top-5 worst CC deviation     : 0.1255
Avg set size                 : 1.676  (std 1.492, min 1, max 13)
Avg distinct superclasses    : 0.000


In [8]:
# ── LAC + MA binary distance (strict_split) ──────────────────────────────────
results_lac_ma_corr = experiments_lac.run_lac_ma_binary_dist_avg_opt_strict_split(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_smaller,
    lamda_values,
)

Marginal coverage            : 0.8998
Mean class-conditional cov.  : 0.8998
Worst-class CC deviation     : 0.1442  (class 35)
Top-5 worst CC deviation     : 0.1210
Avg set size                 : 1.577  (std 1.184, min 1, max 10)
Avg distinct superclasses    : 1.259


In [9]:
# ── LAC + MS binary distance (strict_split) ──────────────────────────────────
results_lac_ms_corr = experiments_lac.run_lac_ms_binary_dist_avg_opt_strict_split(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_ms,
    lamda_values,
)

Marginal coverage            : 0.8983
Mean class-conditional cov.  : 0.8983
Worst-class CC deviation     : 0.1649  (class 10)
Top-5 worst CC deviation     : 0.1203
Avg set size                 : 1.532  (std 1.087, min 1, max 10)
Avg distinct superclasses    : 1.265


In [10]:
# ── LAC + MA superclass mass (strict_split) ──────────────────────────────────
results_lac_ma_sc_corr = experiments_lac.run_lac_ma_superclass_avg_opt_strict_split(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_smaller,
    lamda_values,
)

Marginal coverage            : 0.8582
Mean class-conditional cov.  : 0.8582
Worst-class CC deviation     : 0.2639  (class 35)
Top-5 worst CC deviation     : 0.2141
Avg set size                 : 1.272  (std 0.649, min 1, max 6)
Avg distinct superclasses    : 1.170


## 4. SAPS experiments

In [11]:
# ── Baseline SAPS (no distance penalty) ───────────────────────────────────
# lamda_2 controls rank spacing; 0 = LAC, larger = more spacing between ranks
results_saps_base = experiments_saps.run_saps_baseline(
    probabilities, labels,
    lamda_2=0.2,
)

Marginal coverage            : 0.9017
Mean class-conditional cov.  : 0.9018
Worst-class CC deviation     : 0.1440  (class 10)
Top-5 worst CC deviation     : 0.1208
Avg set size                 : 1.823  (std 1.046, min 1, max 6)
Avg distinct superclasses    : 0.000


In [12]:
# ── SAPS + MS binary distance penalty (strict_split) ─────────────────────────
# lamda_2 fixed; lamda_3 (distance weight) selected on validation fold
lamda_3_values = np.linspace(0, 0.5, 20).tolist()

results_saps_ms_corr = experiments_saps.run_saps_ms_binary_dist_avg_opt_strict_split(
    probabilities, labels,
    adjacency_matrix, adjacency_matrix_ms,
    lamda_2=0.2,
    lamda_3_values=lamda_3_values,
)

Marginal coverage            : 0.8991
Mean class-conditional cov.  : 0.8991
Worst-class CC deviation     : 0.1611  (class 10)
Top-5 worst CC deviation     : 0.1167
Avg set size                 : 1.704  (std 0.957, min 1, max 6)
Avg distinct superclasses    : 1.345


## 5. Compare all results

In [13]:
import pandas as pd

all_results = [
    ('RAPS baseline',          results_raps_base),
    ('RAPS MA orig',           results_raps_ma_orig),
    ('RAPS MA corr',           results_raps_ma_corr),
    ('RAPS MS corr',           results_raps_ms_corr),
    ('RAPS MS fixed lambda',  results_raps_ms_fixed_lambda),
    ('LAC baseline',           results_lac_base),
    ('LAC MA binary corr',     results_lac_ma_corr),
    ('LAC MS binary corr',     results_lac_ms_corr),
    ('LAC MA superclass corr', results_lac_ma_sc_corr),
    ('SAPS baseline',          results_saps_base),
    ('SAPS MS corr',           results_saps_ms_corr),
]

rows = {}
for name, r in all_results:
    rows[name] = {
        'Coverage':      f"{r['marginal_coverage']:.4f}",
        'Mean CC':       f"{r['mean_class_cc']:.4f}",
        'Worst CC dev':  f"{r['worst_cc_deviation']:.4f}",
        'Top-5 CC dev':  f"{r['top5_cc_deviation']:.4f}",
        'Avg set size':  f"{r['set_sizes']['mean']:.3f}",
        'Std set size':  f"{r['set_sizes']['std']:.3f}",
    }

pd.DataFrame(rows).T


,Coverage,Mean CC,Worst CC dev,Top-5 CC dev,Avg set size,Std set size
RAPS baseline,0.9002,0.9002,0.1345,0.1175,2.693,3.950
RAPS MA orig,0.9011,0.9011,0.1294,0.1115,1.955,2.073
RAPS MA corr,0.9014,0.9014,0.1273,0.1140,1.974,2.126
RAPS MS corr,0.9004,0.9004,0.1565,0.1163,1.856,2.014
RAPS MS fixed lambda,0.9017,0.9018,0.1489,0.1102,1.908,2.119
LAC baseline,0.9037,0.9037,0.1796,0.1255,1.676,1.492
LAC MA binary corr,0.8998,0.8998,0.1442,0.1210,1.577,1.184
LAC MS binary corr,0.8983,0.8983,0.1649,0.1203,1.532,1.087
LAC MA superclass corr,0.8582,0.8582,0.2639,0.2141,1.272,0.649
SAPS baseline,0.9017,0.9018,0.1440,0.1208,1.823,1.046
